# 00 - Ollama y embeddings

Objetivo: usar `Ollama` por separado para dos tareas distintas:

- generar texto con un modelo de chat,
- generar embeddings con el modelo de vectores.


In [1]:
import os
from pprint import pprint
from pathlib import Path

from dotenv import load_dotenv
from langchain_ollama import ChatOllama, OllamaEmbeddings

load_dotenv(Path("..").resolve() / ".env")

OLLAMA_PORT = os.getenv("OLLAMA_PORT", "11434")
CHAT_MODEL = os.getenv("CHAT_MODEL", "llama3")
EMBEDDING_MODEL = os.getenv("EMBEDDING_MODEL", "embeddinggemma")
OLLAMA_BASE_URL = f"http://127.0.0.1:{OLLAMA_PORT}"

print({
    "OLLAMA_BASE_URL": OLLAMA_BASE_URL,
    "CHAT_MODEL": CHAT_MODEL,
    "EMBEDDING_MODEL": EMBEDDING_MODEL,
})


{'OLLAMA_BASE_URL': 'http://127.0.0.1:11434', 'CHAT_MODEL': 'llama3', 'EMBEDDING_MODEL': 'embeddinggemma'}


## Hacer `pull` de un modelo desde Python

Si el modelo todavia no existe en Ollama, podemos descargarlo desde codigo usando su API HTTP. Esto es util para automatizar notebooks o preparar el entorno sin salir de Python.


In [2]:
import json
from urllib.request import Request, urlopen

def pull_model(model_name: str, base_url: str = OLLAMA_BASE_URL) -> dict:
    payload = json.dumps({"name": model_name, "stream": False}).encode("utf-8")
    request = Request(
        url=f"{base_url}/api/pull",
        data=payload,
        headers={"Content-Type": "application/json"},
        method="POST",
    )
    with urlopen(request) as response:
        return json.loads(response.read().decode("utf-8"))

# Ejemplo: asegura que el modelo de embeddings este disponible.
# Puedes cambiarlo por CHAT_MODEL si quieres descargar el modelo de chat.
result = pull_model(EMBEDDING_MODEL)
pprint(result)


{'status': 'success'}


In [ ]:
chat_model = ChatOllama(model=CHAT_MODEL, base_url=OLLAMA_BASE_URL, temperature=0)
embedding_model = OllamaEmbeddings(model=EMBEDDING_MODEL, base_url=OLLAMA_BASE_URL)

print(chat_model.invoke("Resume en dos frases para que sirve un sistema RAG.").content)


In [4]:
sentences = [
    "La politica de devoluciones permite cambiar un producto dentro de plazo.",
    "El manual de montaje explica los tornillos y herramientas necesarios.",
    "Qdrant almacena vectores y payload para recuperar contexto relevante.",
]

vectors = embedding_model.embed_documents(sentences)
print(f"Numero de textos: {len(vectors)}")
print(f"Dimension del embedding: {len(vectors[0])}")
print("Primeros 8 valores del primer vector:")
pprint(vectors[0][:8])


Numero de textos: 3
Dimension del embedding: 768
Primeros 8 valores del primer vector:
[-0.07008405,
 -0.050034113,
 0.00072373415,
 0.010708857,
 -0.0140296295,
 0.05033357,
 -0.018281631,
 0.06515346]


Preguntas para discutir:

1. Que diferencia hay entre el modelo de chat y el de embeddings.
2. Por que no reutilizamos directamente texto bruto para buscar similitud.
3. Que dependencias reales necesitamos ya en esta fase.
